# 01 — Grid and audit

This notebook does two things:

1. **Grid**: builds the master 10 m grid, the chunks and the pixel index raster (local, fast).
2. **Audit**: asks Earth Engine which Sentinel-1 images exist over the AOI in the season window (metadata only: **no exports, no cost**).

At the end you choose which **tracks** to process. Background: `docs/01_sar_basics.md` §6–7, `docs/03_pipeline_overview.md`.

> **Safe to re-run:** finished work is skipped. If the kernel dies or a cell crashes, just run the same cells again.

## Setup
Set `CONFIG_PATH` to your config file, then run the cell.

In [ ]:
from pathlib import Path
import sys

# Project root = parent of notebooks/. Adding src/ is only needed if you did not run `pip install -e .`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# >>> Change this to YOUR config file (copied from config/pipeline.example.yaml; it must live in config/) <<<
CONFIG_PATH = ROOT / "config" / "my_aoi_season.yaml"

from sar_pipeline import config, resources
cfg = config.load_config(ROOT / CONFIG_PATH)
print("Config :", CONFIG_PATH)
print("Machine:", resources.detect_resources(cfg).describe())

## Step 1 — Grid and pixel index

One master grid (one CRS, 10 m, snapped corner) so every exported chunk lines up exactly. The pixel index gives each pixel a stable id `pid = row × width + col`.

If you get `GridMismatch`, a grid for this AOI/season already exists with different geometry. **Do not delete it**; restore the settings or use a new `aoi.key`/`season.key`.

In [ ]:
from sar_pipeline import grid, index

gd = grid.build_grid(cfg)
index_path = index.build_pixel_index(cfg, gd)
print(f"Grid: {gd['width']} x {gd['height']} px, {len(gd['chunks'])} chunks, pid dtype {gd['pid_dtype']}")
print("Grid folder:", config.grid_dir(cfg))
print("Pixel index:", index_path)

In [ ]:
import geopandas as gpd

chunks = gpd.read_file(config.grid_dir(cfg) / "chunks.gpkg", layer="chunks")
chunks.sort_values("aoi_frac", ascending=False).head(10)   # good pilot candidates have aoi_frac close to 1

## Step 2 — Audit (Earth Engine metadata only)

This lists every Sentinel-1 image over the AOI, groups slices into acquisitions, finds gaps, computes AOI coverage, rain before each pass and slope statistics, and suggests tracks. The audit folder is named by UTC date; requests to Earth Engine stay small even for very large AOIs.

In [ ]:
from sar_pipeline import auth, audit

auth.init_ee(cfg)
audit_folder = audit.run_audit(cfg)          # use force=True to re-query everything
print("Audit folder:", audit_folder)

In [ ]:
import pandas as pd

pd.read_csv(audit_folder / "track_summary.csv")

In [ ]:
pd.read_csv(audit_folder / "track_recommendation.csv")

In [ ]:
from IPython.display import Markdown, display

display(Markdown((audit_folder / "gaps_report.md").read_text()))

In [ ]:
import json

print(json.dumps(json.loads((audit_folder / "slope_stats.json").read_text()), indent=2))
pd.read_csv(audit_folder / "rain_flags.csv").head(20)

## ⛔ Checkpoint 1 — choose tracks

Read the tables above (and `chunk_track_coverage.csv` for large AOIs). Then **edit your config file** and fill, with your own track ids:

```yaml
s1:
  tracks:
    - {track_id: RO123_ASC, role: primary}
```

Questions to ask yourself:
- Does the primary track cover the whole season (sowing → harvest)?
- Is any gap long enough to miss a growth stage?
- Does a secondary track add real information, or only near-duplicate dates?

Then continue with `02_export.ipynb`.